# Ariketa 3.4: xG ERREALA — Gure neurona StatsBomb datuekin ⚽ (EGITEKO)

Aurreko ariketan **dataset sintetiko bat** erabili dugu.

Han abantaila bat genuen:

> guk geuk ezagutzen genuen datuak sortzeko “arau sekretua” (`z_true`).

Orain jauzi bat egingo dugu:

> **benetako futbol-partidetako jaurtiketak erabiliko ditugu.**

Datu-iturria: **StatsBomb Open Data**.

Ariketa honetan 2018ko gizonezkoen FIFA World Cup-eko partida batzuk erabiliko ditugu.

### Zer aldatu da?

```text
AURREKO ARIKETA
guk sortutako datuak
↓
z_true ezagutzen dugu
↓
neurona entrenatu


ARIKETA HAU
benetako partidako datuak
↓
ez dago guk ezagutzen dugun z_true-rik
↓
neurona entrenatu
↓
datuetatik patroia aurkitu
```

### Gure lehen eredua

Bi input erabiliko ditugu berriro:

```text
x1 = distantzia_m
x2 = zentrotik_angelua
```

eta target-a:

```text
y = gola
```

Ondoren hirugarren input-a gehituko dugu:

```text
x3 = buruz
```

Azkenik gure xG-a StatsBomb-en `statsbomb_xg` probabilitatearekin alderatuko dugu.

> **Garrantzitsua:** `statsbomb_xg` EZ dugu training-erako erabiliko. Amaieran konparatzeko bakarrik.


## 0 — Datu-iturria eta atribuzioa 🟢

StatsBomb Open Data-k datuak JSON fitxategietan eskaintzen ditu:

```text
competitions.json
matches/
events/
lineups/
...
```

Guk bi urrats egingo ditugu:

```text
1. 2018 World Cup-eko partiden zerrenda deskargatu
2. partida bakoitzeko events JSON-a deskargatu
```

Events fitxategietan paseak, jaurtiketak, faltak eta beste ekintza asko daude.

Guk soilik:

```text
type = "Shot"
```

gertaerak gordeko ditugu.

### Atribuzioa

StatsBomb Open Data erabiliz analisia argitaratu edo partekatzen bada, datu-iturria StatsBomb dela adierazi behar da.


# A ATALA — Benetako jaurtiketak prestatu

## A.0 — 2018 World Cup-eko partidak deskargatu 🟢

StatsBomb Open Data-ko identifikatzaileak:

```text
competition_id = 43
season_id      = 3
```

hau da:

```text
FIFA World Cup — 2018
```

Klasean deskarga arinago egiteko lehen **32 partidak** erabiliko ditugu.

> Nahi izanez gero, `N_PARTIDAK = None` jarrita txapelketa osoa erabil daiteke.


In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# StatsBomb Open Data — FIFA World Cup 2018
MATCHES_URL = (
    "https://raw.githubusercontent.com/"
    "hudl/open-data/master/data/matches/43/3.json"
)

# Partiden zerrenda deskargatu
response = requests.get(MATCHES_URL, timeout=30)
response.raise_for_status()
matches = response.json()

# Dataz ordenatu, klasean emaitza egonkorragoa izateko
matches = sorted(matches, key=lambda m: m['match_date'])

print('Txapelketako partidak guztira:', len(matches))

# Klasean deskarga arinagoa egiteko:
# lehen 32 partidak erabiliko ditugu.
#
# Txapelketa osoa erabiltzeko:
# N_PARTIDAK = None
N_PARTIDAK = 32

if N_PARTIDAK is None:
    hautatutako_partidak = matches
else:
    hautatutako_partidak = matches[:N_PARTIDAK]

print('Erabiliko ditugun partidak:', len(hautatutako_partidak))
print(
    'Lehen partida:',
    hautatutako_partidak[0]['match_date'],
    hautatutako_partidak[0]['home_team']['home_team_name'],
    'vs',
    hautatutako_partidak[0]['away_team']['away_team_name']
)


## A.1 — Event-ak deskargatu eta jaurtiketak atera 🟢

Partida bakoitzak milaka event izan ditzake:

```text
Pass
Carry
Duel
Shot
Foul
...
```

Guk `Shot` motako event-ak bakarrik gordeko ditugu.

Gainera, oinarrizko eredua errazago interpretatzeko:

```text
shot.type = "Open Play"
```

jaurtiketak bakarrik erabiliko ditugu.

Jaurtiketa bakoitzetik gordeko dugu:

```text
location            → [x, y]
body_part           → Head / Left Foot / Right Foot / ...
outcome             → Goal / Saved / Off T / ...
statsbomb_xg        → StatsBomb-en xG
player
team
match_id
```


In [ ]:
EVENTS_BASE = (
    "https://raw.githubusercontent.com/"
    "hudl/open-data/master/data/events"
)

jaurtiketak = []

for i, match in enumerate(hautatutako_partidak, start=1):

    match_id = match['match_id']
    url = f"{EVENTS_BASE}/{match_id}.json"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    events = response.json()

    for event in events:

        # Shot event-ak bakarrik
        if event.get('type', {}).get('name') != 'Shot':
            continue

        shot = event.get('shot', {})

        # Open Play jaurtiketak bakarrik:
        # penaltiak eta falta zuzenak kanpoan uzten ditugu.
        if shot.get('type', {}).get('name') != 'Open Play':
            continue

        location = event.get('location')

        # Kokapenik ez badago ezin dugu distantzia/angelua kalkulatu.
        if not location or len(location) < 2:
            continue

        jaurtiketak.append({
            'match_id': match_id,
            'match_date': match['match_date'],
            'team': event.get('team', {}).get('name'),
            'player': event.get('player', {}).get('name'),
            'x': location[0],
            'y': location[1],
            'shot_body_part': shot.get('body_part', {}).get('name'),
            'shot_outcome': shot.get('outcome', {}).get('name'),
            'statsbomb_xg': shot.get('statsbomb_xg')
        })

    if i % 8 == 0 or i == len(hautatutako_partidak):
        print(f'Deskargatuta: {i}/{len(hautatutako_partidak)} partida')

df = pd.DataFrame(jaurtiketak)

print('\nJaurtiketak guztira:', len(df))
df.head()


## A.2 — StatsBomb koordenatuak → gure ezaugarriak 🟢🟡

StatsBomb-ek zelaia honela adierazten du:

```text
x: 0 ────────────────────────────── 120
                                      ATEA
y: 0
   │
   │
  40 ───────────────────────────── atearen erdia
   │
   │
y: 80
```

Erasoko atearen erdigunea:

```text
(120, 40)
```

### Metroetara hurbilketa didaktikoa

StatsBomb-en `120 × 80` koordenatu-sistema pitch estandar batera eramango dugu:

```text
105 m × 68 m
```

Ondoren kalkulatuko dugu:

**1. Distantzia**

$$
d = \sqrt{dx^2 + dy^2}
$$

**2. Zentrotik angelua**

$$
\theta = \arctan\left(\frac{|dy|}{dx}\right)
$$

Gure `zentrotik_angelua`:

```text
0°      → aurrez aurre
handiago → alboago
```

Hau **ez da** atearen bi zutoinen arteko opening-angle klasikoa; aurreko notebook-eko definizio bera mantentzen dugu.

### Target eta burua

```text
shot_outcome == "Goal" → gola = 1
bestela                 → gola = 0

shot_body_part == "Head" → buruz = 1
bestela                  → buruz = 0
```


In [ ]:
# StatsBomb pitch -> 105m x 68m pitch hurbilketa
PITCH_LUZERA_M = 105
PITCH_ZABALERA_M = 68

# StatsBomb attacking goal centre = (120, 40)

# Atearekiko distantzia horizontala (metro aproximatuak)
dx_m = (120 - df['x']) / 120 * PITCH_LUZERA_M

# Zelaiko erditik desbideratze bertikala
dy_m = np.abs(df['y'] - 40) / 80 * PITCH_ZABALERA_M

# bete ____ : Pitagoras
df['distantzia_m'] = np.sqrt(____**2 + ____**2)

# bete ____ :
# 0° = zentral
# handiagoa = alboago
df['zentrotik_angelua'] = np.degrees(
    np.arctan2(____, ____)
)

# Goal -> 1; gainerako outcome-ak -> 0
df['gola'] = (df['shot_outcome'] == ____).astype(int)

# Head -> 1; beste body part-ak -> 0
df['buruz'] = (df['shot_body_part'] == ____).astype(int)

# statsbomb_xg ez duten errenkadak kendu, gero konparatu ahal izateko
df = df.dropna(subset=['statsbomb_xg']).reset_index(drop=True)

df[[
    'x', 'y',
    'distantzia_m',
    'zentrotik_angelua',
    'shot_body_part',
    'gola',
    'statsbomb_xg'
]].head()


## A.3 — Begiratu benetako datuak 🟢

Orain galdera hauek egin:

- Zenbat jaurtiketa ditugu?
- Zenbat izan dira gol?
- Zein da gol-tasa?
- Zenbat dira buruzkoak?
- Nolakoa da StatsBomb xG-ren banaketa?

Gogoratu:

```python
df['gola'].mean()
```

0/1 aldagaia denez, **gol-proportzioa** da.


In [ ]:
print('Jaurtiketak:', len(df))
print('Golak:', df['gola'].sum())
print('Gol-tasa:', round(df['gola'].mean(), 3))
print('Buruzkoak:', df['buruz'].sum())
print('Buruzko %:', round(df['buruz'].mean() * 100, 1))

print('\nBody part-ak:')
print(df['shot_body_part'].value_counts())

print('\nStatsBomb xG:')
print(df['statsbomb_xg'].describe().round(3))


## A.4 — Jaurtiketen kokapena ikusi 🟢

Grafiko bakoitzeko puntu bat = jaurtiketa bat.

Helburua ez da grafiko perfektua egitea, baizik eta datuak zentzuzkoak direla egiaztatzea:

```text
jaurtiketa gehienak aurkariaren atearen inguruan?
oso urruneko jaurtiketak gutxiago?
golak eremu arriskutsuagoetan pilatzen dira?
```


In [ ]:
plt.figure(figsize=(10, 6))

# Puntu bakoitza jaurtiketa bat.
# c = gola erabilita 0/1 emaitzaren arabera bereizten dira.
plt.scatter(
    df['x'],
    df['y'],
    c=df['gola'],
    alpha=0.45
)

plt.xlim(60, 120)
plt.ylim(80, 0)

plt.xlabel('StatsBomb x')
plt.ylabel('StatsBomb y')
plt.title('Open Play jaurtiketen kokapena — StatsBomb')

plt.show()


## A.5 — TRAIN / TEST + StandardScaler 🟡

Gure lehen neuronak:

```text
X = [distantzia_m, zentrotik_angelua]
y = gola
```

erabiliko du.

`statsbomb_xg` **ez da X-en sartzen**.

Zergatik?

> Gure eredua StatsBomb-en xG-a kopiatzeko entrenatuko genuke bestela.

TRAIN / TEST banaketa egin ondoren:

```text
fit_transform(TRAIN)
transform(TEST)
```

erabiliko dugu berriro.

> **TRAIN-ek eskala ikasten du; TEST-ek TRAIN-en eskala erabiltzen du.**


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ezaugarriak = ['distantzia_m', 'zentrotik_angelua']

# DataFrame-a TRAIN / TEST banatu.
# Horrela StatsBomb xG eta buruz zutabeak ere lerro berekin mantenduko dira.
train_df, test_df = train_test_split(
    df,
    test_size=____,
    random_state=42,
    stratify=____
)

X_train = train_df[ezaugarriak].values
X_test = test_df[ezaugarriak].values

y_train = train_df['gola'].values.astype(float)
y_test = test_df['gola'].values.astype(float)

scaler = StandardScaler()

# bete ____ : fit TRAIN-en bakarrik
X_train_s = scaler.____(X_train)

# bete ____ : TEST transform bakarrik
X_test_s = scaler.____(X_test)

print('TRAIN:', len(train_df), '| TEST:', len(test_df))
print('TRAIN gol-tasa:', round(y_train.mean(), 3))
print('TEST gol-tasa :', round(y_test.mean(), 3))


# B ATALA — Gure neurona benetako datuekin

Aurreko ariketako neurona bera erabiliko dugu:

```text
X
↓
z = X @ w + b
↓
sigmoid
↓
p = xG
↓
Log Loss
↓
gradient
↓
w,b eguneratu
```

Ez dago neurona berezirik StatsBomb-entzat.

> **Algoritmo bera; datuak dira berriak.**


## B.0 — Funtzioak berrerabili 🟡

Lau funtzio behar ditugu:

```text
sigmoidea()
aurreikusi()
galera()
gradientea()
```

Gogoratu:

> Sigmoidea **ez da iterazio bakoitzean aldatzen**.

Aldatzen direnak:

```text
w eta b
```

dira.


In [ ]:
def sigmoidea(z):
    return 1 / (1 + np.exp(____))


def aurreikusi(X, w, b):
    z = ____ @ ____ + ____
    return sigmoidea(z)


def galera(y, p):
    p = np.clip(p, 1e-9, 1 - 1e-9)

    return -np.mean(
        y*np.log(____) + (1-y)*np.log(1-____)
    )


def gradientea(X, y, p):
    errorea = ____ - ____

    gw = X.T @ errorea / len(y)
    gb = errorea.mean()

    return gw, gb


## B.1 — Training loop 🟡🔴

Full-batch Gradient Descent erabiliko dugu.

Epoch bakoitzean:

```text
TRAIN guztia
↓
predict
↓
Log Loss
↓
gradient
↓
w eta b eguneratu
```

`2000` epoch erabiliko ditugu aurreko ariketarekin koherente izateko.


In [ ]:
# TODO 🔴
# Entrenatu neurona 2000 epoch:
#
# w = zeros
# b = 0
# eta = 0.1
# galerak = []
#
# epoch bakoitzean:
#   1. predict
#   2. loss gorde
#   3. gradient
#   4. w,b update


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(galerak)

plt.xlabel('Epoch')
plt.ylabel('TRAIN Log Loss')
plt.title('StatsBomb datuak — Gradient Descent')

plt.show()


## B.2 — TEST: probabilitateen kalitatea neurtu 🟢

Orain galdera hau egin behar dugu:

> **Gure ereduak probabilitate erabilgarriak ikasi ditu, ala 0 eta 1 arteko zenbakiak besterik ez ditu sortzen?**

Horretarako gure eredua oso eredu sinple batekin alderatuko dugu.

### 1. Zer da baseline probabilistikoa?

Pentsa TRAIN multzoan:

```text
1000 jaurtiketa
100 gol
```

Orduan:

```text
TRAIN gol-tasa = 100 / 1000 = 0.10
```

**Baseline probabilistikoak** ez du distantzia, angelua edo beste ezaugarririk erabiltzen.

Jaurtiketa GUZTIEI probabilitate bera ematen die:

```text
6 m-ko jaurtiketa   → 0.10
12 m-ko jaurtiketa  → 0.10
30 m-ko jaurtiketa  → 0.10
```

Hau da:

> **“Ez dakit jaurtiketa hau nolakoa den; badakit bakarrik TRAIN datuetan jaurtiketen %10 gola izan direla.”**

Notebook-ean:

```python
p_baseline = np.full(
    shape=len(y_test),
    fill_value=y_train.mean()
)
```

`y_train.mean()` erabiltzen dugu, **ez `y_test.mean()`**, TEST-ek ez duelako informaziorik eman behar eredua edo baseline-a prestatzean.

### 2. Zer espero dugu gure eredutik?

Gure neuronak hau erabiltzen du:

```text
distantzia + angelua
```

Beraz espero dugu jaurtiketa guztiak berdin ez tratatzea.

Adibidez:

| Jaurtiketa | Baseline | Gure eredua |
|---|---:|---:|
| 6 m, zentral | 0.10 | 0.65 |
| 28 m, alboan | 0.10 | 0.03 |
| 12 m, zentral | 0.10 | 0.35 |

Gure eredua baseline-a baino hobea bada, esan nahi du distantzia eta angelua erabiliz informazio erabilgarria ikasi duela.

### 3. Zergatik Log Loss?

xG-ren helburua **probabilitate onak** sortzea da.

Accuracy-k hau galdetzen du:

> “Klasea asmatu al du?”

Log Loss-ek hau galdetzen du:

> “Eman duen probabilitatea zenbateraino izan da ona?”

Futbolean jaurtiketa gehienak ez dira gol. Horregatik:

```text
beti "ez gola" esatea
```

accuracy handia eman dezake, baina xG eredu txarra izango litzateke.

### 4. Nola interpretatu Log Loss?

Ariketa honetan nahikoa da hau gogoratzea:

> **Log Loss txikiagoa = probabilitate hobeak**

Beraz:

```text
gure_loss < baseline_loss
```

bada:

> gure ereduak baseline probabilistikoa gainditu du.


In [ ]:
# Gure neuronaren TEST probabilitateak
p_test = aurreikusi(X_test_s, w, b)

# bete ____ : gure TEST Log Loss
gure_loss = galera(____, ____)

# Baseline:
# TRAIN gol-tasa probabilitate bera jaurtiketa guztientzat
p_baseline = np.full(
    shape=len(y_test),
    fill_value=____
)

baseline_loss = galera(y_test, p_baseline)

# Accuracy ere ikusi
y_pred = (p_test >= 0.5).astype(int)
accuracy = (y_pred == y_test).mean()

# Beti 0 esango bagenu
accuracy_zero = (np.zeros_like(y_test) == y_test).mean()

print('Gure TEST Log Loss:', round(gure_loss, 4))
print('Baseline Log Loss :', round(baseline_loss, 4))
print('Gure accuracy     :', round(accuracy, 3))
print('Beti 0 accuracy   :', round(accuracy_zero, 3))


# C ATALA — Gure xG vs StatsBomb xG

Orain TEST jaurtiketa bakoitzean bi probabilitate ditugu:

```text
gure_xg
statsbomb_xg
```

StatsBomb xG-a **ez dugu training-ean erabili**.

Bi gauza alderatuko ditugu:

```text
1. TEST Log Loss
2. probabilitateen scatter plot-a
```

## Scatter plot-a nola irakurri?

Puntu bakoitza **TEST jaurtiketa bat** da.

```text
X ardatza → StatsBomb xG
Y ardatza → gure xG
```

Adibidez:

```text
StatsBomb xG = 0.20
Gure xG      = 0.25
```

jaurtiketa hori `(0.20, 0.25)` puntuan agertuko da.

### Diagonala: `y = x`

Marra etenak hau adierazten du:

> **Bi ereduek probabilitate bera eman dute.**

Adibidez:

```text
StatsBomb = 0.40
Gurea     = 0.40
```

→ puntua diagonalean.

### Diagonalaren gainean

```text
Gure xG > StatsBomb xG
```

Gure ereduak jaurtiketa arriskutsuagoa dela uste du.

### Diagonalaren azpian

```text
Gure xG < StatsBomb xG
```

StatsBomb-ek jaurtiketa arriskutsuagoa dela uste du.

### Lerrotik distantzia

```text
diagonaletik gertu  → bi ereduak nahiko ados
diagonaletik urrun  → bi ereduen arteko desadostasun handiagoa
```

### Kontuz: scatter plot-ak EZ du esaten zein den hobea

Grafikoak hau erakusten du:

> **bi ereduen arteko adostasuna / desadostasuna**

Baina ez du esaten zein den zuzenagoa.

Horretarako, benetako emaitzak (`y_test`) erabiliz kalkulatutako **TEST Log Loss** konparatzen dugu.

Gogoratzeko:

> **Scatter plot → bi ereduek zenbateraino antzeko probabilitateak ematen dituzten.**  
> **Log Loss → probabilitate horiek benetako emaitzekiko zenbateraino diren onak.**


In [ ]:
# TEST-eko StatsBomb xG
sb_xg_test = test_df['statsbomb_xg'].values.astype(float)

# bete ____ : StatsBomb-en TEST Log Loss
sb_loss = galera(____, ____)

print('Baseline Log Loss :', round(baseline_loss, 4))
print('Gure xG Log Loss  :', round(gure_loss, 4))
print('StatsBomb Log Loss:', round(sb_loss, 4))

# Gure xG vs StatsBomb xG
plt.figure(figsize=(6, 6))
plt.scatter(sb_xg_test, p_test, alpha=0.5)

plt.plot([0, 1], [0, 1], '--')

plt.xlabel('StatsBomb xG')
plt.ylabel('Gure xG')
plt.title('Gure eredua vs StatsBomb')

plt.xlim(0, 1)
plt.ylim(0, 1)

plt.show()


# D ATALA — Hirugarren feature-a: `buruz`

Orain aurreko notebook-eko ideia errealitatean probatuko dugu.

Lehen eredua:

$$
z=b+w_1x_1+w_2x_2
$$

Bigarren eredua:

$$
z=b+w_1x_1+w_2x_2+w_3x_3
$$

non:

```text
x3 = buruz
```

eta:

```text
buruz = 1 → Head
buruz = 0 → bestela
```

Helburua:

> body part informazioa gehitzeak probabilitate-modeloa hobetzen duen ikustea.

Ez dugu `buruz` eskalatuko; 0/1 bezala utziko dugu interpretazioa errazteko.


In [ ]:
# TRAIN / TEST buruz zutabeak
buruz_train = train_df['buruz'].values.reshape(-1, 1)
buruz_test = test_df['buruz'].values.reshape(-1, 1)

# Hirugarren feature-a gehitu
X_train_3 = np.hstack([X_train_s, buruz_train])
X_test_3 = np.hstack([X_test_s, buruz_test])

# TODO 🔴:
# 3 feature-ko neurona entrenatu.
#
# w3 = ...
# b3 = ...
# galerak3 = ...
#
# 2000 epoch
#
# Amaieran kalkulatu:
# p3_test
# loss3
#
# eta konparatu:
# gure_loss vs loss3


# E ATALA — Ondorioak

Bi notebook-en arteko aldea oso garrantzitsua da.

### Dataset sintetikoa

```text
z_true guk sortu
↓
benetako araua ezagutzen genuen
↓
neurona araua berreskuratzen saiatu
```

### StatsBomb datuak

```text
benetako jaurtiketak
↓
ez dugu "true formula" ezagutzen
↓
ereduak patroia datuetatik estimatzen du
```

Hau da Machine Learning-en benetako egoera gehienetan gertatzen dena.

### Gogoratzeko

> **Eredua ez da “egia”. Datuetan aurkitutako erlazioen hurbilketa bat da.**

Eta xG-a:

> **ez da gola / ez-gola aurresateko ziurtasun bat; probabilitate-estimazio bat da.**
